# 08 — Brain states in space and time (material for Figure 1)

The state space and the states as they unfold during a film. Pick the state set in the setup cell
and the cohort, film and state below.

| plot | what |
|---|---|
| 8.0 | latent space, one participant in colour over everyone in grey |
| 8.1 | latent space by film (task) |
| 8.2 | latent space by brain state |
| 8.3 | every viewer's state over one film, and the share of viewers in each state moment by moment |
| 8.4 | one state in detail: the regions most active in it over the film, and its profile against the other states |


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import cohorts as cp

# ---- THE STATE SET: any atlas x window x clusterer stage 4b wrote (listed below)
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "HMM2_pca3_8", "ThresholdCluster_umap3_8", "MeanShift_umap3_3"

# ---- panel C: data balance. "time" = every cohort the same HOURS of scan per draw (whole scans,
# pooled transition counts): people and stimulus duration balanced together. None = cohort means.
BALANCE  = "time"
HOURS    = 10
VERSUS   = "mean"                  # C: each cohort vs the mean of the others; "pairs" = every pair
N_BOOT   = 200                     # draws for panel C, bootstrap over people for panel D
PVALUE   = "stars"                 # Figure 3C confidence column: None, "exact" or "stars"; marker hollow when P(tier) < .95
TEST     = "tier"                  # "tier": P(tier), share of bootstrap |d| in the observed tier (* ≥.90, ** ≥.95, *** ≥.99);
                                    # "difference" (paired t), "tost" (equivalence ±0.2 SD), "both", "all"
COMBINE  = "random"                # "mixed": one mixed model, task share of variance; or ω² per cohort combined: "random", "fixed", "equal"
GUIDES   = None                    # Figure 3 task x person: dashed lines at (.01, .06) with True, or a tuple; None = none
SAVE_DIR = None                    # e.g. "figures/14": PNG + SVG of every figure

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",
    distinct_minutes="min",   # distinct transitions rarefied to the shortest scan (None = as stored)
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",          # Rest, REST1..4 (HCP); every other task is movie
    alpha=0.10,                        # BH-FDR (A.2, films)
    fc_task=1.0, fc_dyn=0.25,          # A.2: |log2FC| for the graph and the dynamics
    edge_top=12,                       # A.1/A.2: strongest arrows drawn
    sesoi_d=0.2,
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; at {ATLAS}, {WINDOW_S:g} s:")
print(", ".join(sets[(sets.atlas == ATLAS) & (sets.window_s.astype(float) == WINDOW_S)]["states"]))
co = ct.Cohorts(cfg)
co.tasks()

In [ ]:
# ---- what the plots show
EMBEDDING   = "umap3"          # "umap3" | "pca3"
MAX_WINDOWS = 150              # windows per person drawn in the latent plots (at random)
WHO         = {"camcan": 1}    # 8.0: {cohort: n-th subject alphabetically, or a subject id}
COHORT      = "camcan"         # 8.3, 8.4: the cohort ...
TASK        = "Movie"          #   ... and the film (a task= value of that cohort)
STATE       = 7                # 8.4: the state shown in detail (0 .. K-1)
N_REGIONS   = 18               # 8.4: regions in the profile (half most, half least active in STATE)
N_TRACES    = 3                # 8.4: regions drawn over time
MAX_MINUTES = 10               # 8.3, 8.4: the first minutes of the film shown (None = all)
lat = ct.load_latents(co, EMBEDDING, MAX_WINDOWS)


#### Plot 8.0 — latent space, one participant


In [ ]:
# Plot 8.0
cp.fig_latents(co, lat, by="single", who=WHO, embedding=EMBEDDING, paths=True).show()


#### Plot 8.1 — latent space by film


In [ ]:
# Plot 8.1
cp.fig_latents(co, lat, by="task", embedding=EMBEDDING).show()


#### Plot 8.2 — latent space by brain state, training cohorts | projected cohorts


In [ ]:
# Plot 8.2 — two panels: the training cohorts together, the projected cohorts together
co_r, lat_r = ct.by_role(co, lat)
cp.fig_latents(co_r, lat_r, by="cluster", embedding=EMBEDDING).show()


#### Plot 8.3 — the states over one film

Left: one row per viewer, one column per 30-s window (its start in film time, the same clock for every viewer); the strip on top is the most common state at each moment. Right: the share of viewers in each state, moment by moment. Vertical bands = the film drives everyone into the same state.

In [ ]:
# Plot 8.3
tl = ct.state_timeline(co, COHORT, TASK)
if MAX_MINUTES:
    tl = tl[tl["stimulus_start_s"] < 60 * MAX_MINUTES]
print(f"{tl['sub'].nunique()} viewers, {tl['stimulus_start_s'].nunique()} windows")
cp.fig_state_timeline(co, tl, f"{cp.cname(COHORT)} · {TASK} · every viewer's state", name=f"state_timeline_{COHORT}_{TASK}");


#### Plot 8.4 — one state in detail

Right: mean activation of each region (rows, clustered) in every state (columns), the regions most and least active in `STATE`; its column framed. Left: the `N_TRACES` regions most active in `STATE` over the film (mean over viewers, band = SD), on the share of viewers in `STATE` (grey). Activation as in the state maps (per-TR, band-passed 0.01–0.1 Hz, z-scored per run); the first read of a cohort's frames is cached.

In [ ]:
# Plot 8.4
F, M, tl = ct.state_activation(co, COHORT, TASK)      # M: the state profiles over the WHOLE film
if MAX_MINUTES:                                        # only the time courses are cut
    F = F[F["t"] < 60 * MAX_MINUTES]
    tl = tl[tl["stimulus_start_s"] < 60 * MAX_MINUTES]
cp.fig_state_detail(co, F, M, tl, STATE, n_regions=N_REGIONS, n_traces=N_TRACES,
                    title=f"State S{STATE} · {cp.cname(COHORT)} · {TASK}");
